In [1]:
# Standard library imports
import os
import sys
import random
import warnings
import math

# Third-party numerical and data handling
import numpy as np
import pandas as pd
import h5py
import cv2
from PIL import Image

# Visualization
import matplotlib.pyplot as plt
from tqdm import tqdm

# Machine learning utilities
from sklearn.metrics import (
    average_precision_score,
    label_ranking_average_precision_score,
    roc_auc_score
)

# PyTorch core
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler

# PyTorch vision
from torchvision import models

# Albumentations
import albumentations as A
from albumentations.core.transforms_interface import ImageOnlyTransform
from albumentations.pytorch import ToTensorV2

# Custom modules (Kaggle inputs)
# sys.path.append("/kaggle/input/asymmetric-loss-dataset")
# sys.path.append("/kaggle/input/data-preprocessing")
# sys.path.append("/kaggle/input/data-transformations")

from losses import AsymmetricLossOptimized
from preprocess import ecg_processing_pipeline, smart_pad_and_resize_ecg
from transformations import CornerCutout, GradientShadow, PaperFoldEffect, BottomBlur

In [2]:
def check_device():
    """
    Check available compute devices and return the best one.
    Priority: CUDA > MPS > CPU
    """
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("✓ CUDA available")
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("✓ MPS (Apple Silicon GPU) available")
    else:
        device = torch.device("cpu")
        print("✗ Using CPU (no GPU acceleration available)")
    
    print(f"\nSelected device: {device}")
    return device

# Check and get device
device = check_device()

✓ MPS (Apple Silicon GPU) available

Selected device: mps


In [3]:
class Head(nn.Module):
    def __init__(self, in_features, hidden_layer, dropout_rate=0.3):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features, hidden_layer),
            nn.BatchNorm1d(hidden_layer),  
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_layer, hidden_layer // 2), 
            nn.BatchNorm1d(hidden_layer // 2),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_layer // 2, 1)
        )
    
    def forward(self, x):
        return self.layers(x)

class MultiHeadEfficientNet(nn.Module):
    def __init__(self, num_conditions=5, hidden_dim=512, dropout_rate=0.3):
        super().__init__()
        
        backbone = models.efficientnet_v2_s(weights="DEFAULT")
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        
        self.backbone = backbone
        
        self.shared_feature_processor = nn.Sequential(
            nn.Linear(in_features, in_features), 
            nn.BatchNorm1d(in_features),
            nn.GELU(), 
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout_rate)
        )
        
        self.heads = nn.ModuleList([
            Head(hidden_dim, hidden_dim // 2, dropout_rate) 
            for _ in range(num_conditions)
        ])
        
    def forward(self, x):
        backbone_feats = self.backbone(x)
        processed_feats = self.shared_feature_processor(backbone_feats)
        outputs = [head(processed_feats) for head in self.heads]
        return torch.cat(outputs, dim=1)  # [batch, num_conditions]

In [ ]:
def load_contours_from_hdf5(filepath='/kaggle/input/ecg-image-contours/contours.h5'):
    """
    Load all contours from HDF5 file back into dictionary format
    """
    contour_dict = {}
    
    with h5py.File(filepath, 'r') as f:
        for img_id in f.keys():
            grp = f[img_id]
            
            contour_dict[img_id] = {
                'contour': grp['contour'][:],  # Load the contour array
                'scale_x': grp.attrs['scale_x'],
                'scale_y': grp.attrs['scale_y'], 
                'half': grp.attrs['half']
            }
    
    return contour_dict

# Usage
try: 
    print(contours["train_000000"])
except Exception as e: 
    contours = load_contours_from_hdf5(filepath=)

In [ ]:
def process_single_img(
                img,
                img_id,
                desired_aspect=0.5,
                target_width=512
                ):
    value = f"train_{str(img_id).zfill(6)}.png"
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    output = ecg_processing_pipeline(input_image = img, 
                                contour_data = contours[value.split(".")[0]])
    resize_output = smart_pad_and_resize_ecg(output, target_size=(int(target_width*desired_aspect), 512), resize_strategy=cv2.INTER_AREA)
    return resize_output

def imread_clean(path):
    img = Image.open(path)
    img = img.convert('RGB')  # Strips metadata
    return np.array(img)

class ECGDataset(Dataset): 
    def __init__(self,
                image_paths, 
                labels_df, 
                transforms=None):
        
        self.image_paths = image_paths
        self.labels_dict = {idx: torch.tensor(row.values, dtype=torch.float32) 
                            for idx, row in labels_df.iterrows()}
        
        self.idx_to_image_id = {}
        for idx, path in enumerate(self.image_paths):
            filename = os.path.basename(path)
            image_id = int(filename.rsplit('_', 1)[-1].split('.')[0])
            self.idx_to_image_id[idx] = image_id
        
        self.transforms = transforms
        
    def __len__(self): 
        return len(self.image_paths)
        
    def __getitem__(self, idx): 
        image_path = self.image_paths[idx]
        index = self.idx_to_image_id[idx]
        label = self.labels_dict[index]
        image = imread_clean(image_path)
        if image is None:
            raise ValueError(f"Failed to load image: {image_path}")
        # convert image using our segmentation pipeline 
        image_conv = process_single_img(
            img=image, 
            img_id=index,
            desired_aspect=0.5, 
            target_width=512
        )
        image_conv = np.stack([image_conv, image_conv, image_conv], axis=-1)
        if self.transforms is not None: 
            image_tens = self.transforms(image=image_conv)["image"]
        else: 
            image_tens = image_conv
        return image_tens, label

In [ ]:
val_transforms = A.Compose([
    A.CLAHE(
        clip_limit=4,
        tile_grid_size=(8, 8),
        p=1.0
    ),
    # Normalization (ImageNet)
    A.Normalize(mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225]),
    # Convert to tensor
    ToTensorV2()
])

In [ ]:
from sklearn.model_selection import train_test_split

labels_df = pd.read_csv("/kaggle/input/bhf-data-science-centre-ecg-challenge/train_final.csv", index_col=0, dtype=int)

with open("/kaggle/input/bhf-reference-files/broken_images_list.txt", "r") as file:
    broken_train_images = [line.strip() for line in file]
with open("/kaggle/input/bhf-reference-files/broken_test_images_list.txt", "r") as file:
    broken_test_images = [line.strip() for line in file]
with open("/kaggle/input/bhf-reference-files/valid_images_list.txt", "r") as file:
    valid_train_images = [line.strip() for line in file]
with open("/kaggle/input/bhf-reference-files/valid_test_images_list.txt", "r") as file:
    valid_test_images = [line.strip() for line in file]

ids = set([int(obj.split(".")[-2][-6:]) for obj in valid_train_images])

labels_df = labels_df.loc[labels_df.index.isin(ids)]

# split train set, using stratification 
X_train, X_test, y_train, y_test = train_test_split(valid_train_images, 
                                                    labels_df, 
                                                    test_size = 0.2,
                                                    random_state = 42, 
                                                    shuffle = True, 
                                                    stratify = labels_df[["CD", "MI", "AF", "STTC", "HYP"]])

In [ ]:
num_workers = 0 if sys.platform == 'darwin' else 4 
print(f"Using num_workers = {num_workers}")

# train_dataset = ECGDataset(
#                     image_paths=X_train, 
#                     labels_df=labels_df, 
#                     transforms=train_transforms, 
#                     )
val_dataset = ECGDataset(
                    image_paths=X_test, 
                    labels_df=labels_df, 
                    transforms=val_transforms, 
                    ) 

# train_dataloader = DataLoader( 
#                         train_dataset,
#                         batch_size=8, 
#                         shuffle=True, 
#                         num_workers=num_workers, 
#                         pin_memory=True if device.type == "cuda" else False)
val_dataloader = DataLoader( 
                        val_dataset,
                        batch_size=8, 
                        shuffle=False, 
                        num_workers=num_workers, 
                        pin_memory=True if device.type == "cuda" else False)

In [ ]:
def compute_ranking_metrics(all_y_true, all_y_pred):
    """Compute AP/AUROC/LRAP ranking metrics. Inputs are numpy arrays."""
    L = all_y_true.shape[1]

    per_label_ap = []
    per_label_auroc = []
    
    for j in range(L):
        # Average Precision
        ap = average_precision_score(all_y_true[:, j], all_y_pred[:, j])
        per_label_ap.append(float(ap))
        
        # AUROC
        try:
            auroc = roc_auc_score(all_y_true[:, j], all_y_pred[:, j])
            per_label_auroc.append(float(auroc))
        except ValueError:
            # Handle case where only one class is present in y_true
            per_label_auroc.append(float('nan'))

    # Micro-averaged metrics
    micro_ap = average_precision_score(all_y_true.reshape(-1), all_y_pred.reshape(-1))
    try:
        micro_auroc = roc_auc_score(all_y_true.reshape(-1), all_y_pred.reshape(-1))
    except ValueError:
        micro_auroc = float('nan')
    
    # Macro-averaged metrics
    macro_ap = sum(per_label_ap) / L
    valid_aurocs = [x for x in per_label_auroc if not np.isnan(x)]
    macro_auroc = sum(valid_aurocs) / len(valid_aurocs) if valid_aurocs else float('nan')
    
    # LRAP
    lrap = label_ranking_average_precision_score(all_y_true, all_y_pred)

    return {
        "per_label_ap": per_label_ap,
        "per_label_auroc": per_label_auroc,
        "macro_ap": float(macro_ap),
        "macro_auroc": float(macro_auroc),
        "micro_ap": float(micro_ap),
        "micro_auroc": float(micro_auroc),
        "lrap": float(lrap),
    }


In [ ]:
checkpoint_path = "/kaggle/input/ecg-efficientnet-bce/pytorch/default/1/checkpoint.pth"
has_checkpoint = False
try: 
    checkpoint = torch.load(checkpoint_path, map_location=torch.device("cpu"), weights_only=False)
    has_checkpoint = True
    current_epoch = checkpoint["epoch"]
    print(f"Loading from checkpoint, last run epoch was {current_epoch}")
except Exception as e: 
    print("No checkpoint found, continuing as default")
    current_epoch = 0
    sys.exit()

In [ ]:
model = MultiHeadEfficientNet(
    num_conditions=5, 
    hidden_dim=512, 
    dropout_rate=0.3
).to(device)

if has_checkpoint: 
    print("Loading model state dict from checkpoint")
    model.load_state_dict(checkpoint["model_state_dict"])

In [ ]:
val_ranking_stats = []
model.eval()
pbar = tqdm(total=len(val_dataloader),
            desc=f"Epoch {epoch} - Validation", 
            unit="batch")
running_val_loss = torch.tensor(0.0, device=device)
total_samples = 0
val_y_true_list = []
val_y_pred_list = []
with torch.inference_mode(): 
    for i, (inputs, labels) in enumerate(val_dataloader): 
        if (i + 1) % 50 == 0 or (i + 1) == len(val_dataloader):
            pbar.n = i + 1
            pbar.refresh()

        inputs = inputs.to(device)
        labels = labels.to(device)
        
        if device.type == "cuda": 
            with autocast('cuda'): 
                outputs = model(inputs)
        else:
            outputs = model(inputs)
            
        y_pred = torch.sigmoid(outputs)
        val_y_pred_list.append(y_pred.cpu())
        val_y_true_list.append(labels.cpu())
        
pbar.close()

all_y_true = torch.cat(val_y_true_list).numpy()
all_y_pred = torch.cat(val_y_pred_list).numpy()

ranking_metrics = compute_ranking_metrics(all_y_true, all_y_pred)